### PHASE 4 — Thursday | Institutional Partnership Risk

---

Compliance Update: For regulatory reporting, default is defined as loans 180+ days past due OR loans with write-off flag. This differs from operational definitions.

`Business Request from Partnership Director:`
"Some of our partner institutions are consistently sending us students who default. I need to know which universities and colleges have the worst track records."

### Start Spark Session

In [1]:
## Import dependencies and create Spark session
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 14:02:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Global Variables

In [12]:
N = 15  # Number of rows to show in results
schema = "edufin_small"  # edufin_small or edufin_national

### Load the data

In [40]:
## Load the identified datasets and create a temporary views for SQL queries
loans = spark.read.csv(f"../datasets/{schema}/loans.csv", header=True)
institutions = spark.read.csv(f"../datasets/{schema}/institutions.csv", header=True)
customers = spark.read.csv(f"../datasets/{schema}/customers.csv", header=True)

loans.createOrReplaceTempView("loans")
institutions.createOrReplaceTempView("institutions")
customers.createOrReplaceTempView("customers")

print("Loans dataset:")
loans.show(n=5)
print("Customers dataset:")
customers.show(n=5)
print("Institutions dataset:")
institutions.show(n=5)

Loans dataset:
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
|      3|       2440|           902|  318341.25|  Defaulted|  11.64999962|                96|      06-03-

Query 4A (BRD): Institutional Volume Analysis

---

- Calculate: Loan volume and total funding per institution.
- Business Purpose: Identify key partners by size.

In [49]:
query = """
SELECT 
    i.institution_id AS `Institution ID`,
    i.institution_name AS `Institution Name`,
    COUNT(*) AS `Loans`,
    CONCAT(ROUND(SUM(l.loan_amount) / 100000.0, 2), ' L') AS `Loan Amount`
FROM institutions i
LEFT JOIN loans l
    ON i.institution_id = l.institution_id
GROUP BY i.institution_id, i.institution_name
ORDER BY `Loans`DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+--------------+----------------------------------+-----+-----------+
|Institution ID|Institution Name                  |Loans|Loan Amount|
+--------------+----------------------------------+-----+-----------+
|3777          |Modern Arts & Science University  |6    |22.62 L    |
|509           |Government Law University         |6    |20.04 L    |
|4766          |Advanced Arts & Science University|5    |18.72 L    |
|2198          |Government Law College            |5    |35.55 L    |
|2337          |Advanced Commerce University      |5    |17.82 L    |
|1323          |Modern Medical University         |5    |15.81 L    |
|4492          |Modern Engineering College        |5    |32.82 L    |
|1728          |State Commerce College            |5    |8.8 L      |
|1622          |National Management College       |5    |22.67 L    |
|4925          |Indian Arts & Science Institute   |5    |9.6 L      |
|624           |Central Engineering University    |5    |18.97 L    |
|2149          |Mode

Query 4B (BRD): Partner Performance Metrics

---

- Calculate: Default rate per university/college.
- Business Purpose: Measure quality of students from each partner.
- Filter: Minimum 50 loans to ensure statistical significance.

In [33]:
query = """
SELECT 
    i.institution_type AS `Institution Type`,
    COUNT(*) AS `Loans`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Default Count`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rate (%)`
FROM institutions i
LEFT JOIN loans l
    ON i.institution_id = l.institution_id
GROUP BY i.institution_type
HAVING COUNT(*) > 50
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*)  DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----------------+-----+-------------+----------------+
|Institution Type|Loans|Default Count|Default Rate (%)|
+----------------+-----+-------------+----------------+
|College         |2301 |202          |8.78%           |
|Institute       |2248 |195          |8.67%           |
|University      |2268 |193          |8.51%           |
+----------------+-----+-------------+----------------+

Time: 0.180 seconds


Query 4C (BRD): Financial Loss Attribution

---

- Calculate: Total value of defaulted loans attributed to each institution.
- Business Purpose: Determine which partnerships are costing EduFin money.

In [39]:
query = """
SELECT 
    i.institution_id AS `Institution ID`,
    i.institution_name AS `Institution Name`,
    CONCAT(ROUND(SUM(l.loan_amount) / 100000.0, 2), ' L') AS `Loan Amount`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L') AS `Default Amount`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) * 100 / SUM(l.loan_amount), 2), '%') AS `Loss Rate (%)`
FROM institutions i
LEFT JOIN loans l
    ON i.institution_id = l.institution_id
GROUP BY i.institution_id, i.institution_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+--------------+------------------------------------+-----------+--------------+-------------+
|Institution ID|Institution Name                    |Loan Amount|Default Amount|Loss Rate (%)|
+--------------+------------------------------------+-----------+--------------+-------------+
|4310          |National Medical University         |18.04 L    |18.04 L       |100.0%       |
|4461          |Central Arts & Science University   |35.04 L    |17.84 L       |50.92%       |
|4318          |Regional Engineering College        |24.86 L    |16.59 L       |66.74%       |
|231           |Indian Medical Institute            |13.24 L    |13.24 L       |100.0%       |
|4519          |Central Commerce College            |20.27 L    |12.92 L       |63.75%       |
|3010          |Indian Law Institute                |16.93 L    |12.02 L       |71.0%        |
|4343          |Government Arts & Science University|15.55 L    |10.36 L       |66.6%        |
|3282          |Indian Management College         

Query 4D: Partnership Tier & Risk Classification

---

- Calculate: Categorize partners (Preferred vs. Probation vs. Blacklist).
- Business Purpose: Automated partnership review status.
- Rules: Default rate > 15% = Blacklist Candidate.